# Notebook Overview — Evaluate Two Models

## Purpose

This notebook performs the **final independent evaluation** of the trained classifiers for the DIP-based AI image detection pipeline using the normalized **25-dimensional Digital Image Processing (DIP) feature vectors**.

The notebook evaluates the two trained models:

* RBF Support Vector Machine (RBF SVM)
* Multi-Layer Perceptron (MLP)

Both models are applied to the **held-out test dataset**, which has not been used during training, validation, or tuning.

---

## Inputs

This notebook uses the following inputs:

* Normalized test dataset:

  * `metadata/vectors/test_feature_vectors_normalized.csv`

* Trained model files:

  * `metadata/models/final_rbf_svm_model.pkl`
  * `metadata/models/final_mlp_model.pkl`

* Project configuration file:

  * `src/project_config.py`

---

## Assumptions

* The project repository is available locally or cloned at runtime
* All paths are controlled through `project_config.py`
* Prior notebooks have generated the required normalized test dataset and trained model files
* The test dataset remains independent and is used only for final evaluation
* All evaluation metrics use the **AI class as the positive class**
* Optional detailed output is controlled by the `VERBOSE` flag
* This notebook does not perform model training, cross-validation, or hyperparameter tuning

---

## What This Notebook Does

### Cell 1 — Startup

* Import required libraries
* Clone the repository if needed
* Import config-defined paths and constants
* Convert config paths to `Path` objects
* Verify required input files

### Cell 2 — Load Normalized Test Data

* Load the normalized test feature-vector CSV
* Display basic dataset shape
* Optionally display columns and sample rows

### Cell 3 — Validate Test Data and Prepare Evaluation Inputs

* Verify metadata columns
* Validate 25 feature columns
* Check for missing values
* Verify class labels and subset values
* Prepare `X_test` and encoded `y_test`

### Cell 4 — Load Trained Models

* Load the final MLP model
* Load the final RBF SVM model

### Cell 5 — Generate Predictions and Probabilities

* Generate predicted labels
* Generate class probabilities
* Extract AI positive-class scores
* Validate prediction and score lengths

### Cell 6 — Compute Final Evaluation Metrics

* Compute Accuracy
* Compute Precision
* Compute Recall
* Compute F1-score
* Compute ROC AUC

### Cell 7 — Generate and Display Confusion Matrices

* Compute confusion matrices for both models
* Build count and normalized confusion matrix tables
* Optionally display tables and plots

### Cell 8 — Generate and Display ROC Curves

* Compute ROC curve points
* Optionally plot ROC curves and preview ROC points

### Cell 9 — Summarize Final Test Results

* Build final results table
* Build report-friendly comparison table
* Sort results by ROC AUC

### Cell 10 — Save Final Evaluation Outputs

* Save final metrics
* Save JSON results
* Save confusion matrices
* Save ROC curve points
* Save final comparison summary

---

## Outputs

All outputs are saved using config-controlled paths under `metadata/results/`:

* `final_test_results.csv`
* `final_test_results.json`
* `final_confusion_matrix_mlp.csv`
* `final_confusion_matrix_rbf_svm.csv`
* `final_roc_points_mlp.csv`
* `final_roc_points_rbf_svm.csv`
* `final_comparison_summary.csv`

---

## Notes

* This notebook evaluates **both trained classifiers** on identical test data
* The held-out test set is not used for model selection or tuning
* Normalization has already been performed in earlier notebooks
* The AI class is treated as the positive class for binary metrics and ROC analysis
* Results support direct comparison between the final MLP and RBF SVM models
* Optional detailed output is controlled through `VERBOSE`

---

**Next step:**  
Use the saved final evaluation results to support feature-level analysis, source-pair analysis, and final project reporting.

In [ ]:
# ============================================================
# Startup (Environment + Path Setup + Verification)
# ============================================================

import os
import sys
import warnings
from pathlib import Path

# ------------------------------------------------------------
# Core libraries
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import pickle
import json

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

from sklearn.preprocessing import LabelEncoder

# ------------------------------------------------------------
# Notebook display control
# ------------------------------------------------------------
VERBOSE = True

if not VERBOSE:
    warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Clone repo into Colab runtime (if needed)
# ------------------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# ------------------------------------------------------------
# Make src/ importable
# ------------------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ------------------------------------------------------------
# Import shared configuration (config-driven)
# ------------------------------------------------------------
from project_config import (
    TEST_NORMALIZED_PATH,
    FINAL_MLP_MODEL_PATH,
    FINAL_RBF_SVM_MODEL_PATH,
    FINAL_TEST_RESULTS_PATH,
    FINAL_TEST_RESULTS_JSON_PATH,
    FINAL_CONFUSION_MATRIX_MLP_PATH,
    FINAL_CONFUSION_MATRIX_RBF_SVM_PATH,
    FINAL_ROC_POINTS_MLP_PATH,
    FINAL_ROC_POINTS_RBF_SVM_PATH,
    FINAL_COMPARISON_SUMMARY_PATH,
    NUM_FEATURES,
    METADATA_COLUMNS,
    AI_LABEL,
    REAL_LABEL,
)

# ------------------------------------------------------------
# Convert config paths to Path objects
# ------------------------------------------------------------
TEST_FEATURE_VECTORS_NORMALIZED_CSV = Path(TEST_NORMALIZED_PATH)

MLP_MODEL_PATH = Path(FINAL_MLP_MODEL_PATH)
RBF_MODEL_PATH = Path(FINAL_RBF_SVM_MODEL_PATH)

FINAL_RESULTS_CSV_PATH = Path(FINAL_TEST_RESULTS_PATH)
FINAL_RESULTS_JSON_PATH = Path(FINAL_TEST_RESULTS_JSON_PATH)

CONFUSION_MATRIX_MLP_PATH = Path(FINAL_CONFUSION_MATRIX_MLP_PATH)
CONFUSION_MATRIX_RBF_PATH = Path(FINAL_CONFUSION_MATRIX_RBF_SVM_PATH)

ROC_POINTS_MLP_PATH = Path(FINAL_ROC_POINTS_MLP_PATH)
ROC_POINTS_RBF_PATH = Path(FINAL_ROC_POINTS_RBF_SVM_PATH)

FINAL_COMPARISON_PATH = Path(FINAL_COMPARISON_SUMMARY_PATH)

# ------------------------------------------------------------
# Ensure output directory exists
# ------------------------------------------------------------
FINAL_RESULTS_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Verify required input files
# ------------------------------------------------------------
print("Verifying required input files...\n")

required_files = [
    TEST_FEATURE_VECTORS_NORMALIZED_CSV,
    MLP_MODEL_PATH,
    RBF_MODEL_PATH
]

missing = [str(f) for f in required_files if not f.exists()]

if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

print("All required input files are present.")

if VERBOSE:
    print(f"Test data        : {TEST_FEATURE_VECTORS_NORMALIZED_CSV}")
    print(f"MLP model        : {MLP_MODEL_PATH}")
    print(f"RBF model        : {RBF_MODEL_PATH}")
    print(f"Results CSV      : {FINAL_RESULTS_CSV_PATH}")
    print(f"Results JSON     : {FINAL_RESULTS_JSON_PATH}")

print("\nStartup complete.")



In [ ]:
# ============================================================
# Cell 2: Load Normalized Test Data
# ============================================================

print("Loading normalized test dataset...\n")

# ------------------------------------------------------------
# Load CSV
# ------------------------------------------------------------
df_test = pd.read_csv(TEST_FEATURE_VECTORS_NORMALIZED_CSV)

# ------------------------------------------------------------
# Basic dataset info (always shown)
# ------------------------------------------------------------
print("Test dataset loaded successfully.\n")
print(f"Shape: {df_test.shape}")

# ------------------------------------------------------------
# Optional detailed output
# ------------------------------------------------------------
if VERBOSE:
    print("\nColumn names:")
    for col in df_test.columns:
        print(col)

    print("\nFirst 5 rows:")
    display(df_test.head())



In [ ]:
# ============================================================
# Cell 3: Validate Test Data and Prepare Evaluation Inputs
# ============================================================

print("Validating test dataset and preparing evaluation inputs...\n")

# ------------------------------------------------------------
# Verify required metadata columns
# ------------------------------------------------------------
missing_metadata_cols = [col for col in METADATA_COLUMNS if col not in df_test.columns]

if missing_metadata_cols:
    raise ValueError(f"Missing required metadata columns: {missing_metadata_cols}")

print("Metadata columns verified.")

# ------------------------------------------------------------
# Identify feature columns
# ------------------------------------------------------------
feature_columns = [col for col in df_test.columns if col not in METADATA_COLUMNS]

print(f"Number of feature columns found: {len(feature_columns)}")

if len(feature_columns) != NUM_FEATURES:
    raise ValueError(
        f"Expected {NUM_FEATURES} feature columns, but found {len(feature_columns)}."
    )

print("Feature count verified.")

# ------------------------------------------------------------
# Check for missing values
# ------------------------------------------------------------
if df_test[METADATA_COLUMNS + feature_columns].isnull().any().any():
    null_counts = df_test[METADATA_COLUMNS + feature_columns].isnull().sum()
    null_counts = null_counts[null_counts > 0]
    raise ValueError(f"Missing values detected:\n{null_counts}")

print("No missing values detected.")

# ------------------------------------------------------------
# Verify class labels
# ------------------------------------------------------------
unique_labels = sorted(df_test["class_label"].unique().tolist())
expected_labels = sorted([AI_LABEL, REAL_LABEL])

print(f"Observed class labels: {unique_labels}")

if unique_labels != expected_labels:
    raise ValueError(
        f"Expected class labels {expected_labels}, but found {unique_labels}."
    )

print("Class labels verified.")

# ------------------------------------------------------------
# Verify subset column
# ------------------------------------------------------------
unique_subsets = sorted(df_test["subset"].unique().tolist())

if unique_subsets != ["test"]:
    raise ValueError(
        f"Expected subset column to contain only ['test'], but found {unique_subsets}."
    )

print("Subset column verified.")

# ------------------------------------------------------------
# Prepare feature matrix and target vectors
# ------------------------------------------------------------
X_test = df_test[feature_columns].copy()
y_labels = df_test["class_label"].to_numpy()

label_encoder = LabelEncoder()
label_encoder.fit([AI_LABEL, REAL_LABEL])
y_test = label_encoder.transform(y_labels)

print("\nEvaluation input arrays prepared successfully.")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

# ------------------------------------------------------------
# Optional detailed output
# ------------------------------------------------------------
if VERBOSE:
    label_mapping = {
        class_name: int(label_encoder.transform([class_name])[0])
        for class_name in label_encoder.classes_
    }

    print("\nEncoded label mapping:")
    for class_name, encoded_value in label_mapping.items():
        print(f"  {class_name} -> {encoded_value}")

    class_counts = df_test["class_label"].value_counts().sort_index()

    print("\nTest set class distribution:")
    for label, count in class_counts.items():
        print(f"  {label}: {count}")



In [ ]:
# ============================================================
# Cell 4: Load Trained Models
# ============================================================

print("Loading trained models...\n")

# ------------------------------------------------------------
# Load MLP model
# ------------------------------------------------------------
with open(MLP_MODEL_PATH, "rb") as f:
    mlp_model = pickle.load(f)

# ------------------------------------------------------------
# Load RBF SVM model
# ------------------------------------------------------------
with open(RBF_MODEL_PATH, "rb") as f:
    rbf_model = pickle.load(f)

print("Models loaded successfully.")

if VERBOSE:
    print(f"\nLoaded MLP model from:     {MLP_MODEL_PATH}")
    print(f"Loaded RBF SVM model from: {RBF_MODEL_PATH}")



In [ ]:
# ============================================================
# Cell 5: Generate Test-Set Predictions and Probabilities
# ============================================================

print("Generating test-set predictions and probabilities...\n")

# ------------------------------------------------------------
# Generate predicted class labels (encoded)
# ------------------------------------------------------------
y_pred_mlp = mlp_model.predict(X_test)
y_pred_rbf = rbf_model.predict(X_test)

print("Predicted class labels generated successfully.")
print(f"Number of MLP predictions: {len(y_pred_mlp)}")
print(f"Number of RBF SVM predictions: {len(y_pred_rbf)}")

# ------------------------------------------------------------
# Generate class probabilities
# ------------------------------------------------------------
if hasattr(mlp_model, "predict_proba"):
    y_proba_mlp = mlp_model.predict_proba(X_test)
else:
    raise AttributeError("The loaded MLP model does not support predict_proba().")

if hasattr(rbf_model, "predict_proba"):
    y_proba_rbf = rbf_model.predict_proba(X_test)
else:
    raise AttributeError(
        "The loaded RBF SVM model does not support predict_proba(). "
        "Ensure probability=True was enabled during training."
    )

print("Predicted class probabilities generated successfully.")

# ------------------------------------------------------------
# Verify class ordering and extract positive-class scores
# ------------------------------------------------------------
positive_class_encoded = label_encoder.transform([AI_LABEL])[0]

if not hasattr(mlp_model, "classes_"):
    raise AttributeError("The loaded MLP model does not expose classes_.")

if not hasattr(rbf_model, "classes_"):
    raise AttributeError("The loaded RBF SVM model does not expose classes_.")

mlp_class_order = list(mlp_model.classes_)
rbf_class_order = list(rbf_model.classes_)

if positive_class_encoded not in mlp_class_order:
    raise ValueError(
        f"Encoded positive class {positive_class_encoded} not found in "
        f"mlp_model.classes_: {mlp_class_order}"
    )

if positive_class_encoded not in rbf_class_order:
    raise ValueError(
        f"Encoded positive class {positive_class_encoded} not found in "
        f"rbf_model.classes_: {rbf_class_order}"
    )

mlp_positive_class_index = mlp_class_order.index(positive_class_encoded)
rbf_positive_class_index = rbf_class_order.index(positive_class_encoded)

y_score_mlp = y_proba_mlp[:, mlp_positive_class_index]
y_score_rbf = y_proba_rbf[:, rbf_positive_class_index]

# ------------------------------------------------------------
# Basic output checks
# ------------------------------------------------------------
if len(y_pred_mlp) != len(y_test):
    raise ValueError(
        f"MLP prediction length mismatch: len(y_pred_mlp)={len(y_pred_mlp)} "
        f"but len(y_test)={len(y_test)}"
    )

if len(y_score_mlp) != len(y_test):
    raise ValueError(
        f"MLP score length mismatch: len(y_score_mlp)={len(y_score_mlp)} "
        f"but len(y_test)={len(y_test)}"
    )

if len(y_pred_rbf) != len(y_test):
    raise ValueError(
        f"RBF SVM prediction length mismatch: len(y_pred_rbf)={len(y_pred_rbf)} "
        f"but len(y_test)={len(y_test)}"
    )

if len(y_score_rbf) != len(y_test):
    raise ValueError(
        f"RBF SVM score length mismatch: len(y_score_rbf)={len(y_score_rbf)} "
        f"but len(y_test)={len(y_test)}"
    )

print("Prediction length checks passed.")

# ------------------------------------------------------------
# Optional detailed output
# ------------------------------------------------------------
if VERBOSE:
    print(f"\nPositive class for ROC scoring: '{AI_LABEL}' -> encoded {positive_class_encoded}")
    print(f"MLP class order: {mlp_class_order}")
    print(f"MLP positive class index: {mlp_positive_class_index}")

    print(f"\nRBF SVM class order: {rbf_class_order}")
    print(f"RBF SVM positive class index: {rbf_positive_class_index}")

    print("\nFirst 10 MLP predicted labels (encoded):")
    print(y_pred_mlp[:10].tolist())

    print("\nFirst 10 MLP positive-class probabilities:")
    print(np.round(y_score_mlp[:10], 6).tolist())

    print("\nFirst 10 RBF SVM predicted labels (encoded):")
    print(y_pred_rbf[:10].tolist())

    print("\nFirst 10 RBF SVM positive-class probabilities:")
    print(np.round(y_score_rbf[:10], 6).tolist())



In [ ]:
# ============================================================
# Cell 6: Compute Final Evaluation Metrics
# ============================================================

print("Computing final evaluation metrics...\n")

# ------------------------------------------------------------
# Define positive class for binary metrics
# ------------------------------------------------------------
positive_class_encoded = label_encoder.transform([AI_LABEL])[0]

# Create binary target for ROC-AUC with AI as positive class = 1
y_test_ai_positive = (df_test["class_label"] == AI_LABEL).astype(int).to_numpy()

# ------------------------------------------------------------
# Compute classification metrics (MLP)
# ------------------------------------------------------------
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)

precision_mlp = precision_score(
    y_test,
    y_pred_mlp,
    pos_label=positive_class_encoded
)

recall_mlp = recall_score(
    y_test,
    y_pred_mlp,
    pos_label=positive_class_encoded
)

f1_mlp = f1_score(
    y_test,
    y_pred_mlp,
    pos_label=positive_class_encoded
)

roc_auc_mlp = roc_auc_score(y_test_ai_positive, y_score_mlp)

# ------------------------------------------------------------
# Compute classification metrics (RBF SVM)
# ------------------------------------------------------------
accuracy_rbf = accuracy_score(y_test, y_pred_rbf)

precision_rbf = precision_score(
    y_test,
    y_pred_rbf,
    pos_label=positive_class_encoded
)

recall_rbf = recall_score(
    y_test,
    y_pred_rbf,
    pos_label=positive_class_encoded
)

f1_rbf = f1_score(
    y_test,
    y_pred_rbf,
    pos_label=positive_class_encoded
)

roc_auc_rbf = roc_auc_score(y_test_ai_positive, y_score_rbf)

# ------------------------------------------------------------
# Display final metric values
# ------------------------------------------------------------
print("Final test-set evaluation metrics computed.")

if VERBOSE:
    print("\nMLP:")
    print(f"  Accuracy : {accuracy_mlp:.4f}")
    print(f"  Precision: {precision_mlp:.4f}")
    print(f"  Recall   : {recall_mlp:.4f}")
    print(f"  F1-score : {f1_mlp:.4f}")
    print(f"  ROC-AUC  : {roc_auc_mlp:.4f}\n")

    print("RBF SVM:")
    print(f"  Accuracy : {accuracy_rbf:.4f}")
    print(f"  Precision: {precision_rbf:.4f}")
    print(f"  Recall   : {recall_rbf:.4f}")
    print(f"  F1-score : {f1_rbf:.4f}")
    print(f"  ROC-AUC  : {roc_auc_rbf:.4f}")

# ------------------------------------------------------------
# Store metrics in dictionaries for later use
# ------------------------------------------------------------
final_metrics_mlp = {
    "model": "MLP",
    "accuracy": float(accuracy_mlp),
    "precision": float(precision_mlp),
    "recall": float(recall_mlp),
    "f1_score": float(f1_mlp),
    "roc_auc": float(roc_auc_mlp)
}

final_metrics_rbf = {
    "model": "RBF SVM",
    "accuracy": float(accuracy_rbf),
    "precision": float(precision_rbf),
    "recall": float(recall_rbf),
    "f1_score": float(f1_rbf),
    "roc_auc": float(roc_auc_rbf)
}



In [ ]:
# ============================================================
# Cell 7: Generate and Display Confusion Matrices
# ============================================================

print("Generating confusion matrices...\n")

# ------------------------------------------------------------
# Define class order for confusion matrices
# ------------------------------------------------------------
class_names = [AI_LABEL, REAL_LABEL]
class_order_encoded = label_encoder.transform(class_names)

# ------------------------------------------------------------
# Compute confusion matrices
# ------------------------------------------------------------
cm_mlp = confusion_matrix(y_test, y_pred_mlp, labels=class_order_encoded)
cm_rbf = confusion_matrix(y_test, y_pred_rbf, labels=class_order_encoded)

print("Confusion matrices computed successfully.")

# ------------------------------------------------------------
# Create raw confusion matrix tables
# ------------------------------------------------------------
cm_mlp_df = pd.DataFrame(
    cm_mlp,
    index=[f"Actual: {label}" for label in class_names],
    columns=[f"Predicted: {label}" for label in class_names]
)

cm_rbf_df = pd.DataFrame(
    cm_rbf,
    index=[f"Actual: {label}" for label in class_names],
    columns=[f"Predicted: {label}" for label in class_names]
)

# ------------------------------------------------------------
# Compute normalized confusion matrices (row-wise)
# ------------------------------------------------------------
cm_mlp_normalized = cm_mlp.astype(float) / cm_mlp.sum(axis=1, keepdims=True)
cm_rbf_normalized = cm_rbf.astype(float) / cm_rbf.sum(axis=1, keepdims=True)

cm_mlp_norm_df = pd.DataFrame(
    cm_mlp_normalized,
    index=[f"Actual: {label}" for label in class_names],
    columns=[f"Predicted: {label}" for label in class_names]
)

cm_rbf_norm_df = pd.DataFrame(
    cm_rbf_normalized,
    index=[f"Actual: {label}" for label in class_names],
    columns=[f"Predicted: {label}" for label in class_names]
)

# ------------------------------------------------------------
# Optional detailed display
# ------------------------------------------------------------
if VERBOSE:
    print("\nMLP Confusion Matrix (counts):")
    display(cm_mlp_df)

    print("\nRBF SVM Confusion Matrix (counts):")
    display(cm_rbf_df)

    # ------------------------------------------------------------
    # Plot side-by-side confusion matrices (counts)
    # ------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    sns.heatmap(
        cm_mlp,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[0]
    )
    axes[0].set_title("MLP Confusion Matrix")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")

    sns.heatmap(
        cm_rbf,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[1]
    )
    axes[1].set_title("RBF SVM Confusion Matrix")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Actual")

    plt.tight_layout()
    plt.show()

    print("\nMLP Confusion Matrix (normalized):")
    display(cm_mlp_norm_df.round(4))

    print("\nRBF SVM Confusion Matrix (normalized):")
    display(cm_rbf_norm_df.round(4))

    # ------------------------------------------------------------
    # Plot side-by-side normalized confusion matrices
    # ------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    sns.heatmap(
        cm_mlp_normalized,
        annot=True,
        fmt=".3f",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[0]
    )
    axes[0].set_title("MLP Normalized Confusion Matrix")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")

    sns.heatmap(
        cm_rbf_normalized,
        annot=True,
        fmt=".3f",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[1]
    )
    axes[1].set_title("RBF SVM Normalized Confusion Matrix")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("Actual")

    plt.tight_layout()
    plt.show()



In [ ]:
# ============================================================
# Cell 8: Generate and Display ROC Curves
# ============================================================

print("Generating ROC curves...\n")

# ------------------------------------------------------------
# Create binary target with AI as positive class = 1
# ------------------------------------------------------------
y_test_ai_positive = (df_test["class_label"] == AI_LABEL).astype(int).to_numpy()

# ------------------------------------------------------------
# Compute ROC curves
# ------------------------------------------------------------
fpr_mlp, tpr_mlp, thresholds_mlp = roc_curve(y_test_ai_positive, y_score_mlp)
fpr_rbf, tpr_rbf, thresholds_rbf = roc_curve(y_test_ai_positive, y_score_rbf)

print("ROC curves computed successfully.")

if VERBOSE:
    print(f"Number of MLP ROC points: {len(fpr_mlp)}")
    print(f"Number of RBF SVM ROC points: {len(fpr_rbf)}")

    # ------------------------------------------------------------
    # Plot ROC curves (comparison)
    # ------------------------------------------------------------
    plt.figure(figsize=(7, 6))

    plt.plot(
        fpr_mlp,
        tpr_mlp,
        linewidth=2,
        label=f"MLP (AUC = {roc_auc_mlp:.4f})"
    )

    plt.plot(
        fpr_rbf,
        tpr_rbf,
        linewidth=2,
        label=f"RBF SVM (AUC = {roc_auc_rbf:.4f})"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        linewidth=1,
        label="Random Classifier"
    )

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve Comparison (Test Set)")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # ------------------------------------------------------------
    # Preview first few ROC points
    # ------------------------------------------------------------
    roc_preview_mlp = pd.DataFrame({
        "threshold": thresholds_mlp[:10],
        "fpr": fpr_mlp[:10],
        "tpr": tpr_mlp[:10]
    })

    roc_preview_rbf = pd.DataFrame({
        "threshold": thresholds_rbf[:10],
        "fpr": fpr_rbf[:10],
        "tpr": tpr_rbf[:10]
    })

    print("\nFirst 10 ROC points (MLP):")
    display(roc_preview_mlp.round(6))

    print("\nFirst 10 ROC points (RBF SVM):")
    display(roc_preview_rbf.round(6))



In [ ]:
# ============================================================
# Cell 9: Summarize Final Test Results in Tabular Form
# ============================================================

print("Building final test-results summary tables...\n")

# ------------------------------------------------------------
# Create final results summary table
# ------------------------------------------------------------
df_final_results = pd.DataFrame([
    {
        "model": "MLP",
        "model_file": "final_mlp_model.pkl",
        "accuracy": accuracy_mlp,
        "precision": precision_mlp,
        "recall": recall_mlp,
        "f1_score": f1_mlp,
        "roc_auc": roc_auc_mlp
    },
    {
        "model": "RBF SVM",
        "model_file": "final_rbf_svm_model.pkl",
        "accuracy": accuracy_rbf,
        "precision": precision_rbf,
        "recall": recall_rbf,
        "f1_score": f1_rbf,
        "roc_auc": roc_auc_rbf
    }
])

# ------------------------------------------------------------
# Sort by ROC-AUC
# ------------------------------------------------------------
df_final_results = df_final_results.sort_values(
    by="roc_auc",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Create report-friendly comparison summary
# ------------------------------------------------------------
df_final_comparison = pd.DataFrame({
    "metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "MLP": [
        accuracy_mlp,
        precision_mlp,
        recall_mlp,
        f1_mlp,
        roc_auc_mlp
    ],
    "RBF SVM": [
        accuracy_rbf,
        precision_rbf,
        recall_rbf,
        f1_rbf,
        roc_auc_rbf
    ]
})

print("Final test-results summary tables created.")

if VERBOSE:
    print("\nFinal test-results summary table:\n")
    display(df_final_results.round(4))

    print("\nFinal comparison summary:\n")
    display(df_final_comparison.round(4))



In [ ]:
# ============================================================
# Cell 10: Save Final Evaluation Outputs
# ============================================================

print("Saving final evaluation outputs...\n")

# ------------------------------------------------------------
# Save final metric summary table
# ------------------------------------------------------------
df_final_results.to_csv(FINAL_RESULTS_CSV_PATH, index=False)

# ------------------------------------------------------------
# Save final metrics dictionary as JSON
# ------------------------------------------------------------
final_metrics = {
    "MLP": final_metrics_mlp,
    "RBF_SVM": final_metrics_rbf
}

with open(FINAL_RESULTS_JSON_PATH, "w") as f:
    json.dump(final_metrics, f, indent=4)

# ------------------------------------------------------------
# Save confusion matrices
# ------------------------------------------------------------
cm_mlp_df.to_csv(CONFUSION_MATRIX_MLP_PATH)
cm_rbf_df.to_csv(CONFUSION_MATRIX_RBF_PATH)

# ------------------------------------------------------------
# Save ROC curve points
# ------------------------------------------------------------
df_roc_mlp = pd.DataFrame({
    "threshold": thresholds_mlp,
    "fpr": fpr_mlp,
    "tpr": tpr_mlp
})

df_roc_rbf = pd.DataFrame({
    "threshold": thresholds_rbf,
    "fpr": fpr_rbf,
    "tpr": tpr_rbf
})

df_roc_mlp.to_csv(ROC_POINTS_MLP_PATH, index=False)
df_roc_rbf.to_csv(ROC_POINTS_RBF_PATH, index=False)

# ------------------------------------------------------------
# Save final comparison summary
# ------------------------------------------------------------
df_final_comparison.to_csv(FINAL_COMPARISON_PATH, index=False)

# ------------------------------------------------------------
# Optional detailed output
# ------------------------------------------------------------
if VERBOSE:
    print(f"Saved final results CSV:            {FINAL_RESULTS_CSV_PATH}")
    print(f"Saved final results JSON:           {FINAL_RESULTS_JSON_PATH}")
    print(f"Saved MLP confusion matrix CSV:     {CONFUSION_MATRIX_MLP_PATH}")
    print(f"Saved RBF SVM confusion matrix CSV: {CONFUSION_MATRIX_RBF_PATH}")
    print(f"Saved MLP ROC curve points CSV:     {ROC_POINTS_MLP_PATH}")
    print(f"Saved RBF ROC curve points CSV:     {ROC_POINTS_RBF_PATH}")
    print(f"Saved comparison summary CSV:       {FINAL_COMPARISON_PATH}")

print("\nAll final evaluation outputs saved successfully.")

